In [1]:
import pandas as pd
import numpy as np
import os

Statistical models (linear mixed models, with individual as a random effect) were fitted to model the relationships between (a) temperature and chirp duration, (b) temperature and carrier frequency. From these two fitted models, the predicted relationship is:

chDur $= 10678.825 - 133.595 * temp$

carrier_freq $= 3991.6939 + 66.5529 * temp$

So, for every degree increase in temperature, chirp duration is expected to decrease by 133.6 and carrier frequency to increase by 66.6.

The regression equations provides a direct way to adjust for temperature. To standardise everything to a mean temperature, for example 29.1, one needs to calculate the differen between the observed temperatuere and the mean temperature and then adjust the chirp duration and carrier frequency by the required amount.

Say chDur = 7427, carrier_freq = 5641, at temp = 28.2, then 29.1 - 28.2 = 0.9. So chDur should be decreased by 0.9 * 133.595 = 120.2355 i.e. the new chDur should by 7427 - 120.235 = 7306.764. The carrier_freq should be increased by 0.9 * 66.55 = 59.895 i.e. the new carrier_freq should be 5641 + 59.895 = 5701.597.

This produces the "chirp_duration_target" and "carrier_freq_target" parameters need to adjust spectrograms using the **librosa** audio processing library.

In [2]:
class TemperatureCorrection:

    def __init__(self, p_raw_temp, p_temp):
        self.p_raw_temp = p_raw_temp
        self.p_temp = p_temp

    def correct_temperature(self):
        df_raw_temp = pd.read_excel(self.p_raw_temp)
        df_temp = pd.read_csv(p_temp)
        mean_temp = np.round(df_raw_temp["Temperature"].mean(),1)
        df_temp["mean_temp"] = [mean_temp]*len(df_temp)
        df_temp["temp_diff"] = df_temp["mean_temp"]-df_temp["temp"]
        df_temp["chDur_decrease"] = df_temp["temp_diff"]*133.595
        df_temp["carrier_increase"] = df_temp["temp_diff"]*66.55
        df_temp["chDur_target"] = df_temp["chDur"]-df_temp["chDur_decrease"]
        df_temp["carrier_target"] = df_temp["carrier_freq"] + df_temp["carrier_increase"]
        df_temp["time_sace_factor"] = df_temp["chDur_target"]/df_temp["chDur"]
        df_temp["freq_scale_factor"] = df_temp["carrier_target"]/df_temp["carrier_freq"]

        return df_temp

In [3]:
p_raw_temp = "recording_temperature_data.xlsx"
p_temp = "temperature_data.csv"

In [4]:
tc = TemperatureCorrection(p_raw_temp, p_temp)

In [5]:
df = tc.correct_temperature()

In [6]:
df

,chirp,ID,chDur,carrier_freq,temp,mean_temp,temp_diff,chDur_decrease,carrier_increase,chDur_target,carrier_target,time_sace_factor,freq_scale_factor
0,LPL11_0,LPL,5412.0,5727.832031,32.2,28.6,-3.6,-480.942,-239.58,5892.942,5488.252031,1.088866,0.958173
1,LPL11_2,LPL,5498.0,5534.033203,32.2,28.6,-3.6,-480.942,-239.58,5978.942,5294.453203,1.087476,0.956708
2,LPL11_6,LPL,5486.0,5620.166016,32.2,28.6,-3.6,-480.942,-239.58,5966.942,5380.586016,1.087667,0.957371
3,LPL11_8,LPL,5508.0,5620.166016,32.2,28.6,-3.6,-480.942,-239.58,5988.942,5380.586016,1.087317,0.957371
4,LPL11_9,LPL,5499.0,5534.033203,32.2,28.6,-3.6,-480.942,-239.58,5979.942,5294.453203,1.087460,0.956708
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1094,RBB13_256,RBB,6657.0,5727.832031,27.0,28.6,1.6,213.752,106.48,6443.248,5834.312031,0.967891,1.018590
1095,RBB13_258,RBB,6827.0,5684.765625,27.0,28.6,1.6,213.752,106.48,6613.248,5791.245625,0.968690,1.018731
1096,RBB13_259,RBB,6672.0,5878.564453,27.0,28.6,1.6,213.752,106.48,6458.248,5985.044453,0.967963,1.018113
1097,RBB13_260,RBB,6612.0,5598.632812,27.0,28.6,1.6,213.752,106.48,6398.248,5705.112813,0.967672,1.019019


In [7]:
df.to_csv("corrected_temperature_data.csv", index = False)